# S4 · Evaluation of the balance forecasts

We evaluate 30-day forecasts for **all 500 accounts**. We apply the same ±€1,000 cap to every candidate and keep the observed balances intact.

- **Original windows.** Three 30-day backtests in October, November, and December 2022. These give the headline result. The original results informed the study design, so we report them as a retrospective backtest.
- **Earlier replay.** Five 30-day backtests from December 2021 through April 2022. We tune the policy on windows before that period and report this result separately.

## 1 · What is the overall result for the project delivery?

We report **122.75% overall historical MAPE** for the delivered policy across the 500 accounts and three original evaluation windows. We pool every scored account-day. We also report 96.27% in the earlier replay, whose settings were selected on its own preceding windows. The submission contains 15,000 daily predictions for 1 to 30 January 2023. **January MAPE is unavailable** because the corresponding actual balances are unavailable.

In [1]:
from datetime import date, timedelta
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
from IPython.display import HTML, display
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype"
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "parameters.yml").is_file())
results = root / "data/results/medium/2022-12-31"
scores = pl.read_json(results / "evaluation_scores.json")
windows = pl.read_json(results / "evaluation_windows.json")
model_groups = pl.read_json(results / "model_group_scores.json")

ORIGINAL = "Original 2022 windows"
REPLAY = "Earlier chronological replay"
METRICS = ["MAPE (%)", "MAE (€)", "WAPE (%)"]
threshold = 200
display(scores.filter(pl.col("Selected")).select(
    "study", "MAPE days", *METRICS,
).with_columns(pl.selectors.float().round(2)))

submission = pl.read_csv(results / "submission.csv", schema_overrides={"ACCOUNT_ID": pl.String})
dates = [(date(2022, 12, 31) + timedelta(days=day)).isoformat() for day in range(1, 31)]
assert submission.height == submission["ACCOUNT_ID"].n_unique() == 500
assert submission.columns == ["ACCOUNT_ID", *dates]
assert submission.select(pl.exclude("ACCOUNT_ID").is_finite().all()).row(0) == (True,) * 30
print(f"January submission · {submission.height} accounts · {len(dates)} days · {submission.height * len(dates):,} forecasts")

study,MAPE days,MAPE (%),MAE (€),WAPE (%)
str,i64,f64,f64,f64
"""Original 2022 windows""",45000,122.75,20668.18,98.28
"""Earlier chronological replay""",75000,96.27,13465.82,97.53


January submission · 500 accounts · 30 days · 15,000 forecasts


## 2 · What does the zero control reveal about MAPE?

We forecast zero for every account-day as a control. On each nonzero actual balance, the absolute percentage error is exactly **100%**, regardless of the balance size. A forecast of €0.10 for an actual balance of €0.01 has a 9-cent error and a 900% percentage error. This weighting lets a few small balances dominate MAPE. We exclude exact zeros from MAPE and include them in the euro-error calculations.

We plot MAPE beside mean absolute error (MAE) on the same account-days to test whether the metrics give the same ranking. In the original windows, zero wins on MAPE **100.00% versus 122.75%**, even though its mean error is **€21,030.34 per account-day** versus **€20,668.18** for the selected ensemble. In the earlier replay the ensemble improves both metrics.

In [2]:
control = scores.filter(pl.col("Selected") | (pl.col("model") == "Zero control"))
studies = [ORIGINAL, REPLAY]
names = ["Original windows", "Earlier replay"]
fig = make_subplots(rows=1, cols=2, subplot_titles=("MAPE (%)", "MAE (€ per account-day)"))

for label, subset, color in (
    ("Zero control", control.filter(pl.col("model") == "Zero control"), "#747D89"),
    ("Selected Chronos ensemble", control.filter(pl.col("Selected")), "#147D83"),
):
    values = pl.DataFrame({"study": studies}).join(subset, on="study", validate="1:1")
    for column, metric, units in ((1, "MAPE (%)", "%"), (2, "MAE (€)", "€")):
        y = values[metric].to_list()
        labels = [f"{v:.2f}%" if units == "%" else f"€{v:,.0f}" for v in y]
        fig.add_bar(
            x=names, y=y, text=labels, textposition="outside", cliponaxis=False,
            name=label, legendgroup=label, showlegend=column == 1, marker_color=color,
            row=1, col=column,
        )

fig.update_yaxes(range=[0, 145], ticksuffix="%", row=1, col=1)
fig.update_yaxes(range=[0, 24500], tickprefix="€", separatethousands=True, row=1, col=2)
fig.update_layout(
    barmode="group", template="plotly_white", height=440,
    title="Forecasting zero can win on MAPE",
    legend=dict(orientation="h", y=-0.20),
    margin=dict(l=60, r=25, t=90, b=82),
)
fig.show(config={"responsive": True, "displaylogo": False})

### Where did the largest percentage errors come from?

We traced the largest uncapped persistence error to an account forecast at €133,909.54 from the 31 October 2022 origin. On 25 November its observed balance fell from €204,493.55 to −€2.29, where it stayed through the end of December. We retained those observations because the source and prepared values agree and the cause remains unresolved. [S0 reviews the event](S0__preprocessing.ipynb).

We report the contribution of that episode and the sub-euro observations to the last-balance baseline. The denominator is all 45,000 scored original account-days. This shows what clipping removes from the numerator and how small denominators continue to affect MAPE.

In [3]:
diagnostic = pl.read_json(results / "mape_diagnostic.json").row(0, named=True)
fields = [
    "Episode contribution to uncapped MAPE (pp)",
    "Episode contribution to capped MAPE (pp)",
    "Scored days below €1 in absolute balance",
    "Days below €1 contribution to capped MAPE (pp)",
]
display(pl.DataFrame({
    "Last-balance diagnostic": fields,
    "Value": [f"{diagnostic[name]:,.2f}" if isinstance(diagnostic[name], float)
              else str(diagnostic[name]) for name in fields],
}))

Last-balance diagnostic,Value
str,str
"""Episode contribution to uncapp…","""779.69"""
"""Episode contribution to capped…","""5.84"""
"""Scored days below €1 in absolu…","""76"""
"""Days below €1 contribution to …","""410.17"""


## 3 · Why do the results differ by customer group?

We assign each account to a group using only balances available at that forecast origin. As in S2, G01 is the quieter group, G02 has the highest share of negative balances, and G03 is the more active of the other two groups. Membership can change over time. Each forecast day belongs to exactly one group, and the three groups cover all 500 accounts in every evaluation window.

We report each group's balance scale, sub-euro frequency and forecast errors. We split the original-window MAPE bars by the absolute size of the actual balance. Each contribution uses the full group denominator. We also recalculate MAPE on the €1-and-up subset as a sensitivity check. The official scores use all observed nonzero balances.

In [4]:
selected_groups = model_groups.filter(pl.col("Selected")).with_columns(
    (100 * pl.col("Sub-euro days") / pl.col("MAPE days")).alias("Sub-euro days (%)"),
)
for study in (ORIGINAL, REPLAY):
    print(study)
    with pl.Config(tbl_cols=9, tbl_rows=5, fmt_str_lengths=50):
        display(selected_groups.filter(pl.col("study") == study).select(
            "group", "MAPE days", "Median |balance| (€)", "Sub-euro days (%)",
            *METRICS, "MAPE at €1 or more (%)",
        ).with_columns(pl.selectors.float().round(2)))

original_groups = selected_groups.filter(pl.col("study") == ORIGINAL).sort("group")
below_one = original_groups["Sub-euro MAPE contribution (pp)"]
one_to_five = original_groups["€1 to under €5 MAPE contribution (pp)"]
fig = go.Figure()
for label, contribution, color in (
    ("€5 or more", original_groups["MAPE (%)"] - below_one - one_to_five, "#147D83"),
    ("€1 to under €5", one_to_five, "#DB842E"),
    ("Below €1", below_one, "#B55041"),
):
    fig.add_bar(
        x=original_groups["group"].to_list(), y=contribution.to_list(),
        name=label, marker_color=color,
    )
for group, total in original_groups.select("group", "MAPE (%)").iter_rows():
    fig.add_annotation(x=group, y=total, text=f"{total:.2f}%", showarrow=False, yshift=12)
fig.add_hline(y=100, line_dash="dot", line_color="#747D89", annotation_text="Zero control")
fig.update_layout(
    barmode="stack", template="plotly_white", height=430,
    title="Where group MAPE comes from · Original windows",
    yaxis=dict(title="Contribution to group MAPE (percentage points)", range=[0, 220]),
    legend=dict(orientation="h", y=-0.14),
    margin=dict(l=80, r=35, t=60, b=77),
)
fig.show(config={"responsive": True, "displaylogo": False})

group,MAPE days,Median |balance| (€),Sub-euro days (%),MAPE (%),MAE (€),WAPE (%),MAPE at €1 or more (%)
str,i64,f64,f64,f64,f64,f64,f64
"""G01""",13440,2406.9,0.01,182.49,16010.84,96.35,169.19
"""G02""",15300,170.96,0.44,98.28,392.71,92.93,98.27
"""G03""",16260,1327.5,0.04,96.41,43596.19,98.93,96.28


group,MAPE days,Median |balance| (€),Sub-euro days (%),MAPE (%),MAE (€),WAPE (%),MAPE at €1 or more (%)
str,i64,f64,f64,f64,f64,f64,f64
"""G01""",22020,2054.96,0.15,93.95,12499.02,95.68,93.77
"""G02""",25920,159.76,0.62,101.54,505.52,93.34,98.75
"""G03""",27060,1185.96,0.1,93.1,26666.84,98.33,91.86


Original 2022 windows
Earlier chronological replay


We found that **G01's 182.49% MAPE** was concentrated in low-balance days. In the original windows, 80 out of 13,440 group observations had absolute balances below €5. They contributed **97.32 percentage points**, over half the group's MAPE. G02 had 67 sub-euro observations but their errors added only 0.44 percentage points. G02 also had the smallest typical balances and the lowest MAE at **€392.71**.

G03 had the lowest original-window MAPE at **96.41%** and the largest euro errors at **€43,596.19 MAE**. Its observed balances averaged €44,068 in absolute value, and its WAPE was 98.93%. In the earlier replay, G01 scored 93.95% MAPE and G02 scored 101.54%. We interpret the group differences through the balance scale and the concentration of large relative errors in each study.

## 4 · How did all models score?

We report **all 29 S3 candidates and controls** from both studies. We present pooled and group-level MAPE, MAE and WAPE in expandable tables. We sort each table by the pooled metric. The overall column combines account-days for all 500 accounts and weights each metric by its own denominator. A group leader found retrospectively can motivate another experiment.

We keep the original-window scatter to show the error tradeoff across all candidates. Each point uses the same ±€1,000 cap. Values left of the dashed line meet the pooled 200% requirement in those windows. The hover labels report the model name and the share of predictions we clipped.

<details>
<summary>Forecasting methods behind all 29 labels</summary>

We use the labels below in the scorecards. We fit or apply each base method to each account's own history. For S3, we form ensembles and apply calibration before clipping the final predictions. The SMA, weekly persistence and drift rules use forward-filled history. The calendar regressions use up to 365 days of valid daily changes.

| Scorecard label | Forecasting method |
|---|---|
| `Last balance` | Persistence. We repeat the most recent observed balance for all 30 days. |
| `SMA 7` | 7-day simple moving average. We forecast one constant level from the last seven daily balances. |
| `SMA 14` | 14-day simple moving average. We forecast one constant level from the last 14 daily balances. |
| `SMA 28` | 28-day simple moving average. We forecast one constant level from the last 28 daily balances. |
| `Seasonal naive 7` | Weekly seasonal persistence. We cycle through the last seven daily balances. |
| `Local level` | Simple exponential smoothing. We fit a constant forecast level using up to 365 days and a fixed level gain of 0.20. |
| `Damped Holt` | Holt's damped trend method. We use up to 365 days, a level gain of 0.20, a trend gain of 0.05 and damping of 0.90. |
| `Drift 90` | Linear drift. We extrapolate the slope between the origin balance and the balance 90 days earlier. |
| `AR(1) changes` | First-order autoregression on daily balance changes. We forecast changes and accumulate them from the last level. |
| `Weekday changes` | Ridge regression on daily changes with weekday indicators. We accumulate the predicted changes. |
| `Calendar changes` | Ridge regression on daily changes with weekday indicators and an account-specific monthly inflow window when available. We accumulate the predicted changes. |
| `Two-part calendar` | Logistic regression for movement probability and ridge regression for signed size on movement days, using calendar features. We multiply the predictions and accumulate the expected changes. |
| `Chronos-Bolt Tiny` | Zero-shot median forecast from Amazon's pretrained `amazon/chronos-bolt-tiny` checkpoint. We use up to 512 daily history positions per account. |
| `Chronos-2 Small` | Zero-shot median forecast from the pretrained `autogluon/chronos-2-small` checkpoint. We use up to 512 daily history positions per account. |
| `Zero control` | Constant zero. We forecast zero on every day. |
| `MAPE median 30` | Flat weighted-median forecast from the last 30 calendar days. We weight finite, nonzero balances by the inverse of their absolute value. |
| `MAPE median 90` | Flat weighted-median forecast from the last 90 calendar days. We weight finite, nonzero balances by the inverse of their absolute value. |
| `MAPE median 365` | Flat weighted-median forecast from the last 365 calendar days. We weight finite, nonzero balances by the inverse of their absolute value. |
| `Mean classical` | Equal-weight mean of Last balance, SMA 14 and Local level. |
| `Mean MAPE medians` | Equal-weight mean of MAPE median 30, MAPE median 90 and MAPE median 365. |
| `Mean two Chronos` | Equal-weight mean of the two Chronos median forecasts. |
| `Mean Chronos and persistence` | Equal-weight mean of the two Chronos median forecasts and Last balance. |
| `Mean mixed` | Equal-weight mean of Last balance, Chronos-2 Small and MAPE median 365. |
| `Median mixed` | Pointwise median of Last balance, Chronos-2 Small and MAPE median 365. |
| `Chronos quantile pool (MAPE)` | We pool nine quantiles from each Chronos model and take a weighted median for each day. We weight each quantile by the inverse of its absolute value, with a €0.01 denominator floor. |
| `Calibrated MAPE median 365` | MAPE median 365 with the balance-scale calibration described below. |
| `Calibrated Mean MAPE medians` | Mean MAPE medians with balance-scale calibration. |
| `Calibrated Median mixed` | Median mixed with balance-scale calibration. |
| `Calibrated Chronos quantile pool (MAPE)` | Chronos quantile pool (MAPE) with balance-scale calibration. We used this ensemble for the January delivery after applying the shared cap. |

For **balance-scale calibration**, we group account-origins by the median absolute balance over the last 90 calendar days, with band boundaries at €100, €1,000 and €10,000. Within each band we fit a multiplier between 0 and 1 on the earlier selection windows to minimize MAPE. We apply the fitted multiplier on the later windows. We tune the final cap separately from these model definitions when comparing shared and group-specific caps.

</details>

In [5]:
overall = scores.select(
    "study", "model", *METRICS, pl.lit("All 500").alias("group"),
)
scorecard = overall.vstack(model_groups.select(overall.columns))

for study in (ORIGINAL, REPLAY):
    cohort = scorecard.filter(pl.col("study") == study)
    for metric in METRICS:
        wide = cohort.pivot(on="group", index="model", values=metric)
        columns = [name for name in ("All 500", "G01", "G02", "G03", "Review") if name in wide.columns]
        table = wide.select("model", *columns).sort("All 500", "model", nulls_last=True)
        html = table.to_pandas().to_html(index=False, float_format=lambda value: f"{value:,.2f}")
        display(HTML(f"<details><summary><strong>{study} · {metric}</strong></summary>{html}</details>"))

model,All 500,G01,G02,G03
Zero control,100.00,100.00,100.00,100.00
Calibrated MAPE median 365,101.16,110.48,101.24,93.37
Calibrated Median mixed,103.36,118.29,99.71,94.45
Calibrated Mean MAPE medians,104.47,116.38,100.56,98.29
MAPE median 365,106.51,119.39,113.42,89.35
MAPE median 90,112.59,132.75,110.46,97.93
Mean MAPE medians,114.91,128.28,117.63,101.29
Calibrated Chronos quantile pool (MAPE),122.75,182.49,98.28,96.41
MAPE median 30,133.95,134.97,150.08,117.94
Chronos quantile pool (MAPE),150.97,196.98,136.53,126.53


model,All 500,G01,G02,G03
Chronos-2 Small,"20,538.88","15,888.30",286.19,"43,439.86"
Mean two Chronos,"20,541.06","15,888.40",289.46,"43,442.76"
Median mixed,"20,549.16","15,896.13",297.71,"43,451.01"
Mean Chronos and persistence,"20,550.33","15,892.37",303.89,"43,451.53"
Chronos-Bolt Tiny,"20,553.36","15,894.69",304.90,"43,457.03"
Mean mixed,"20,555.77","15,900.69",307.17,"43,456.61"
Chronos quantile pool (MAPE),"20,570.92","15,910.55",323.08,"43,475.44"
SMA 28,"20,572.10","15,902.61",338.08,"43,471.15"
SMA 14,"20,574.73","15,903.59",338.49,"43,477.22"
Local level,"20,576.62","15,903.03",341.82,"43,479.79"


model,All 500,G01,G02,G03
Chronos-2 Small,97.66,95.61,67.72,98.57
Mean two Chronos,97.67,95.61,68.50,98.58
Median mixed,97.71,95.66,70.45,98.60
Mean Chronos and persistence,97.72,95.63,71.91,98.60
Chronos-Bolt Tiny,97.73,95.65,72.15,98.61
Mean mixed,97.74,95.68,72.69,98.61
Chronos quantile pool (MAPE),97.82,95.74,76.45,98.65
SMA 28,97.82,95.69,80.00,98.64
SMA 14,97.83,95.70,80.10,98.66
Local level,97.84,95.70,80.89,98.66


model,All 500,G01,G02,G03
Calibrated Chronos quantile pool (MAPE),96.27,93.95,101.54,93.10
Calibrated Mean MAPE medians,99.94,103.73,99.90,96.88
Calibrated Median mixed,99.97,96.05,104.23,99.07
Zero control,100.00,100.00,100.00,100.00
Calibrated MAPE median 365,102.03,108.26,100.46,98.46
MAPE median 365,111.83,119.16,118.94,99.07
MAPE median 90,113.91,117.09,121.23,104.32
Mean MAPE medians,122.04,130.08,134.79,103.29
MAPE median 30,159.71,162.64,201.04,117.73
Chronos quantile pool (MAPE),181.46,256.47,181.10,120.76


model,All 500,G01,G02,G03
Chronos-2 Small,"13,354.61","12,387.90",421.47,"26,529.55"
Mean two Chronos,"13,356.84","12,388.84",424.72,"26,531.86"
Median mixed,"13,364.71","12,398.05",430.23,"26,540.89"
Mean Chronos and persistence,"13,367.52","12,394.53",439.02,"26,543.13"
Chronos-Bolt Tiny,"13,369.74","12,399.60",439.53,"26,544.67"
Mean mixed,"13,370.55","12,401.78",441.20,"26,543.53"
SMA 28,"13,381.50","12,406.16",463.02,"26,549.42"
SMA 14,"13,382.26","12,408.48",459.36,"26,553.14"
Chronos quantile pool (MAPE),"13,383.62","12,412.18",444.29,"26,568.33"
Local level,"13,387.27","12,410.09",465.56,"26,559.80"


model,All 500,G01,G02,G03
Chronos-2 Small,96.72,94.83,77.82,97.83
Mean two Chronos,96.74,94.84,78.42,97.83
Median mixed,96.80,94.91,79.43,97.87
Mean Chronos and persistence,96.82,94.88,81.06,97.88
Chronos-Bolt Tiny,96.83,94.92,81.15,97.88
Mean mixed,96.84,94.93,81.46,97.88
SMA 28,96.92,94.97,85.49,97.90
SMA 14,96.92,94.99,84.81,97.91
Chronos quantile pool (MAPE),96.93,95.01,82.03,97.97
Local level,96.96,95.00,85.96,97.94


In [6]:
primary = scores.filter(pl.col("study") == "Original 2022 windows").with_columns(
    pl.when(pl.col("Selected")).then(pl.lit("Selected on earlier windows"))
    .when(pl.col("model") == "Zero control").then(pl.lit("Zero control"))
    .otherwise(pl.lit("Other candidates")).alias("Role"),
)
fig = px.scatter(
    primary.to_pandas(), x="MAPE (%)", y="WAPE (%)", color="Role", symbol="Role",
    color_discrete_map={
        "Selected on earlier windows": "#147D83",
        "Zero control": "#BE5D36",
        "Other candidates": "#8996A7",
    },
    hover_name="model", hover_data={"Capped forecasts (%)": ":.2f"},
    title="Original windows · All forecasts capped at ±€1,000",
)
fig.update_traces(marker_size=11)
fig.add_vline(x=threshold, line_dash="dash", line_color="#BE5D36", annotation_text="200% limit")
fig.update_layout(
    template="plotly_white", height=415, legend=dict(orientation="h", y=-0.19),
    margin=dict(l=65, r=30, t=60, b=80),
)
fig.show(config={"responsive": True, "displaylogo": False})

We found that the calibrated 365-day MAPE median was the strongest history-based method on original-window MAPE at **101.16%**. The calibrated Chronos ensemble was selected using preceding windows and led the earlier replay at **96.27%**. These rankings show why we keep the selection step separate from the evaluation.

## 5 · Did the selected policy clear 200% in every window?

We check each 30-day window because pooled MAPE can hide a failing month. We plot observed MAPE and a conservative stress bound for one account spending seven consecutive days at a balance of magnitude €1. We mark each forecast origin with the last date of information available to the model. The worst observed score was 149.88% and the worst stress bound was 196.60%.

In [7]:
chosen = windows.filter(pl.col("Selected")).with_columns(pl.col("origin").str.to_date()).sort("origin")
plot = chosen.unpivot(
    on=["MAPE (%)", "Stress MAPE bound (%)"], index=["study", "origin"],
    variable_name="Measure", value_name="Percent",
)
fig = px.scatter(
    plot.to_pandas(), x="origin", y="Percent", color="Measure", symbol="study",
    color_discrete_sequence=["#147D83", "#DB842E"],
    title="Selected policy · Observed and stressed MAPE by origin",
    labels={"origin": "Forecast origin", "Percent": "MAPE (%)", "study": "Study"},
)
fig.update_traces(marker_size=10)
fig.add_hline(y=100, line_dash="dot", line_color="#8996A7", annotation_text="Zero control")
fig.add_hline(y=threshold, line_dash="dash", line_color="#BE5D36", annotation_text="200% limit")
fig.update_layout(
    template="plotly_white", height=420, legend=dict(orientation="h", y=-0.23),
    margin=dict(l=62, r=35, t=65, b=95),
)
fig.show(config={"responsive": True, "displaylogo": False})

## 6 · Did group-specific caps improve the forecast?

We tuned one shared cap and three group-specific caps on preceding windows, using the same Chronos forecast in both cases. We compare the later pooled errors and show the MAPE change separately for each group. In the original windows, caps of €1,250, €1,250 and €750 raised MAPE from 122.75% to 126.29%. The stress bound reached 215.21%, and WAPE improved by 0.01 percentage points. In the earlier replay both procedures selected €1,000 for every group. We kept the shared cap for January.

In [8]:
caps = pl.read_json(results / "cap_comparison.json")
groups = pl.read_json(results / "group_cap_scores.json")
with pl.Config(tbl_rows=12, tbl_cols=10, fmt_str_lengths=36):
    display(caps.select(
        "study", "policy", "G01", "G02", "G03", "MAPE (%)", "Worst window MAPE (%)", "WAPE (%)",
    ).with_columns(pl.selectors.float().round(2)))
    display(groups.select("study", "group", "policy", "MAPE (%)").pivot(
        on="policy", index=["study", "group"], values="MAPE (%)",
    ).sort("study", "group").with_columns(pl.selectors.float().round(2)))

study,policy,G01,G02,G03,MAPE (%),Worst window MAPE (%),WAPE (%)
str,str,f64,f64,f64,f64,f64,f64
"""Original 2022 windows""","""Tuned shared cap""",1000.0,1000.0,1000.0,122.75,149.88,98.28
"""Original 2022 windows""","""Tuned group caps""",1250.0,1250.0,750.0,126.29,156.83,98.27
"""Earlier chronological replay""","""Tuned shared cap""",1000.0,1000.0,1000.0,96.27,101.37,97.53
"""Earlier chronological replay""","""Tuned group caps""",1000.0,1000.0,1000.0,96.27,101.37,97.53


study,group,Tuned shared cap,Tuned group caps
str,str,f64,f64
"""Earlier chronological replay""","""G01""",93.95,93.95
"""Earlier chronological replay""","""G02""",101.54,101.54
"""Earlier chronological replay""","""G03""",93.1,93.1
"""Original 2022 windows""","""G01""",182.49,193.35
"""Original 2022 windows""","""G02""",98.28,98.23
"""Original 2022 windows""","""G03""",96.41,97.27


## 7 · What did Chronos and clipping contribute?

We compare three steps under the same cap. The individual Chronos forecasts use the median. The quantile ensemble pools both models' predictive quantiles and weights values nearer zero more heavily to target percentage error. We then learn a scale multiplier for each historical balance band using earlier windows. The first table shows how these steps change MAPE and includes the calibrated historical median as a classical benchmark.

In the second table, we compare the same selected forecast before and after clipping. This isolates the cap's effect on MAPE and euro accuracy. About one third of the original-window forecasts crossed the cap. The 200% requirement and the selection stress limits favored this protection against large percentage errors.

In [9]:
variants = [
    "Chronos-2 Small", "Chronos-Bolt Tiny", "Chronos quantile pool (MAPE)",
    "Calibrated Chronos quantile pool (MAPE)", "Calibrated MAPE median 365",
]
comparison = (
    scores.filter(pl.col("model").is_in(variants))
    .select("model", "study", "MAPE (%)")
    .pivot(on="study", index="model", values="MAPE (%)")
)
with pl.Config(tbl_cols=6, fmt_str_lengths=52):
    display(pl.DataFrame({"model": variants}).join(
        comparison, on="model", validate="1:1",
    ).select("model", ORIGINAL, REPLAY).with_columns(pl.selectors.float().round(2)))
    display(pl.read_json(results / "selected_cap_effect.json").select(
        "study", "Treatment", *METRICS,
    ).sort("study", "Treatment").with_columns(pl.selectors.float().round(2)))

model,Original 2022 windows,Earlier chronological replay
str,f64,f64
"""Calibrated MAPE median 365""",101.16,102.03
"""Calibrated Chronos quantile pool (MAPE)""",122.75,96.27
"""Chronos quantile pool (MAPE)""",150.97,181.46
"""Chronos-2 Small""",288.13,483.54
"""Chronos-Bolt Tiny""",389.05,427.33


study,Treatment,MAPE (%),MAE (€),WAPE (%)
str,str,f64,f64,f64
"""Earlier chronological replay""","""Before clipping""",84.85,3104.96,22.49
"""Earlier chronological replay""","""Shared cap""",96.27,13465.82,97.53
"""Original 2022 windows""","""Before clipping""",896.84,2208.73,10.5
"""Original 2022 windows""","""Shared cap""",122.75,20668.18,98.28


## Conclusion

**We used the groups to diagnose where forecasts work and where they fail.** S1 identified three customer behaviours, and S2 used them to test model routing. In S4, the breakdown exposed concentrated low-balance errors in G01 and high euro errors in G03 despite its relatively low MAPE. Routing different models in S2 increased pilot MAE. Group caps increased original-window MAPE from 122.75% to 126.29%, while the earlier replay selected €1,000 for all three groups. We use the groups to interpret errors and keep one forecast policy for the submission.

**We found that the Chronos gains depended on how we used its forecasts.** On the earlier replay, quantile pooling brought capped MAPE down to 181.46% from over 400% for either individual Chronos model. Balance-scale calibration reduced it further to 96.27%, ahead of the best classical result of 99.94%. The calibrated classical median led the history-based methods on original-window MAPE at 101.16%. The two comparisons support the use of distributional forecasts and calibration for this objective. The contribution of pretraining remains unmeasured in this comparison.

**We met the historical MAPE requirement at a substantial cost in euros.** In the original windows, the ±€1,000 cap reduced the selected forecast's MAPE from 896.84% to **122.75%** and increased WAPE from 10.50% to 98.28%. The cap raised both errors in the earlier replay. We retained it because the capped policy stayed below 200% in all eight historical windows. We report 122.75% as the overall historical MAPE for the policy used in the January delivery. The January score requires actual balances.

**We conclude that MAPE is unreliable as a standalone measure for these balances.** The zero control scored 100% MAPE and ranked first in the original windows, with a mean error of €21,030 per account-day. We propose **WAPE as the primary metric for euro accuracy**, with MAE and group sample sizes alongside it. WAPE divides total absolute error by total absolute observed balance, avoiding a separate division by each small actual. Large balances carry more influence, which makes the per-group breakdown useful. We retain MAPE as the required assignment metric.